# Photo Pipeline v3 — 2-Call Sonnet

**Call 1 (Sonnet)** → factual observation + SEO metadata (name, description, caption, keywords, location, slug)  
**Call 2 (Sonnet)** → artistic analysis for search & recommendations (mood, style_fingerprint, photographic_tradition, compositional_tension, light_quality)

In [ ]:
# Cell: Setup & Schema

# ── imports ──────────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(override=True)

import duckdb, anthropic
import base64, hashlib, time, json as _json, re, sys
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# ── constants ─────────────────────────────────────────────────────────────────
DB_PATH = "./outputs/photos.duckdb"
RAW_DIR = Path("./data/raw")

# ── model selection ───────────────────────────────────────────────────────────
MODEL = "claude-sonnet-4-6"  # used for both calls

# ── target selection ──────────────────────────────────────────────────────────
# Set to None to process all photos, or a list of filenames:
#   PROCESS_PHOTOS = ["Kinan.Sweidan-21.JPG"]
PROCESS_PHOTOS = None

# ── DB + schema ───────────────────────────────────────────────────────────────
con = duckdb.connect(DB_PATH)

con.execute("""
    CREATE TABLE IF NOT EXISTS v3_runs (
        run_id     INTEGER PRIMARY KEY,
        notes      VARCHAR,
        c1_prompt  VARCHAR,
        c2_prompt  VARCHAR,
        created_at TIMESTAMP DEFAULT current_timestamp
    )
""")

con.execute("""
    CREATE TABLE IF NOT EXISTS v3_outputs (
        photo_id              VARCHAR,
        run_id                INTEGER NOT NULL DEFAULT 1,

        -- Call 1 (Sonnet) — factual + SEO
        c1_raw                VARCHAR,
        name                  VARCHAR,
        description           VARCHAR,
        caption               VARCHAR,
        keywords              VARCHAR,
        content_location      VARCHAR,
        slug                  VARCHAR,
        confidence            INTEGER,
        c1_latency_ms         INTEGER,

        -- Call 2 (Sonnet) — artistic
        c2_raw                VARCHAR,
        mood                  VARCHAR,
        light_quality         VARCHAR,
        compositional_tension VARCHAR,
        photographic_tradition VARCHAR,
        style_fingerprint     VARCHAR,
        c2_latency_ms         INTEGER,

        created_at            TIMESTAMP DEFAULT current_timestamp,
        PRIMARY KEY (photo_id, run_id)
    )
""")

# ── load photos ───────────────────────────────────────────────────────────────
def file_hash(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()[:16]

hash_to_path = {
    file_hash(str(p)): p
    for p in sorted(p for ext in ("*.jpg", "*.JPG", "*.jpeg", "*.JPEG")
                    for p in RAW_DIR.glob(ext))
}

if PROCESS_PHOTOS is None:
    target_photos = hash_to_path
else:
    target_names = set(PROCESS_PHOTOS)
    target_photos = {k: v for k, v in hash_to_path.items() if v.name in target_names}
    missing = target_names - {v.name for v in target_photos.values()}
    if missing:
        print(f"  [warn] not found in RAW_DIR: {missing}")

CURRENT_RUN_ID = None  # set by running the New Run cell

# ── summary ───────────────────────────────────────────────────────────────────
runs   = con.execute("SELECT run_id, notes FROM v3_runs ORDER BY run_id").fetchall()
counts = con.execute(
    "SELECT run_id, COUNT(*) FROM v3_outputs GROUP BY run_id ORDER BY run_id"
).fetchall()

print(f"Photos found: {len(hash_to_path)}  |  Targeted: {len(target_photos)}")
if runs:
    print(f"v3_runs: {[f'#{r} {n}' for r, n in runs]}")
print(f"v3_outputs rows: {counts or 'empty'}")


In [ ]:
# Cell: Prompts

# ── Call 1: Factual observation + SEO ────────────────────────────────────────
# Well-tuned prompt — do not change field names or format instructions.
CALL1_PROMPT = """These are black and white environmental portraits and street photographs taken in Chicago, Illinois. The subject is always a person or group of people. Do not describe chromatic colors (red, blue, green, yellow, orange, purple, brown, golden, etc.) — white, black, and grey are valid tonal descriptors.

Look at this photograph and describe only what you literally see.

Write a single dense paragraph. Cover:
- Primary subject: who or what is in the foreground? What are they doing? Describe expression, posture, clothing material and texture, and any held objects.
- Secondary elements: mid-ground and background — name specific objects, people, vehicles, architecture.
- Location markers: look carefully for street signs, business names, transit signage, recognizable buildings or landmarks. Name them explicitly if visible.
- Frame: Portrait, landscape, or square? Describe the tonal range — is it high contrast (deep blacks, bright whites) or soft and low contrast?
- Light: direction (front, side, back/rim), quality (harsh and directional vs soft and diffuse), where highlights fall, depth of shadows.
- Texture & shape: describe fabric weave, skin texture, architectural surfaces. Note strong shapes, silhouettes, and geometric forms.

Be specific and literal. Do not interpret or editorialize.

Then, using only what you described above, generate the following metadata fields. Output them one per line in exactly this format — no preamble, no explanation.

BANNED words (never use these in any field): street, urban, city, candid, scene, moment, photography, photo, monochrome, bustling, hustle, colorful, vibrant, warm tones, cool tones, golden hour, vivid

NAME: Wire-service title — subject with specific clothing detail, clear action verb, location or object if present. Minimum 6 words.

DESCRIPTION: One sentence under 125 characters. Format: [who with specific clothing color] [doing what] [specific where].

CAPTION: Exactly two sentences, factual only. Sentence 1: subject, specific clothing or objects, action. Sentence 2: background — specific architecture, signage, vehicles, or infrastructure.

KEYWORDS: Exactly 12 comma-separated values in this order:
  - 3 subject-descriptor: physical appearance only (hair, clothing material, texture, accessories)
  - 3 action: what the subject is physically doing (e.g. walking, laughing, looking upward)
  - 3 location-identifier: specific place names, architecture type, or Chicago if no specific marker is visible
  - 3 visual-property: tonal and optical qualities (e.g. high contrast, rim lighting, soft shadows, deep blacks, grainy texture, low key, portrait orientation)
Use lowercase. Prefer specific over generic.

CONTENTLOCATION: Use Chicago, IL as the default. Only specify a neighborhood (e.g. River North, The Loop, Wicker Park) or street if you can confirm it from a visible sign, landmark, or street name in the frame.

SLUG: Lowercase hyphenated slug: subject-clothing-action-location. No genre words.

CONFIDENCE: How confident are you in the accuracy of these fields?
  5 — subject details clear, location confirmed from a visible marker in the frame
  4 — subject details clear, location inferred from context (Chicago default is acceptable)
  3 — subject details mostly clear, one or more fields uncertain
  2 — partial description only, multiple fields uncertain
  1 — could not reliably complete the fields
Output the integer only (1-5).

NAME: value
DESCRIPTION: value
CAPTION: value
KEYWORDS: kw1, kw2, kw3, kw4, kw5, kw6, kw7, kw8, kw9, kw10, kw11, kw12
CONTENTLOCATION: Chicago, IL or specific neighborhood
SLUG: value
CONFIDENCE: value"""


# ── Call 2: Artistic analysis for search & recommendations ───────────────────
# Receives the image again + Call 1 output as factual grounding.
# Focused entirely on interpretation — no SEO, no factual fields.
CALL2_PROMPT = """You are an expert in 20th-century street photography with deep knowledge of Cartier-Bresson, Vivian Maier, Daido Moriyama, and the black-and-white humanist tradition.

You will receive a black and white street photograph and a factual description that has already been written about it.
Use that description as your grounding — do not contradict it and do not repeat it verbatim.

Your job is artistic interpretation only. Think through:
- What is the decisive moment or tension in this image?
- How does the tonal rendering serve the mood?
- What is the relationship between figure and environment?
- What photographic tradition or aesthetic does this most closely resemble?

Return each field on its own line in FIELD: value format. No preamble. No explanation.

mood: exactly 3 adjectives, comma-separated (do not repeat words already in the factual description)
light_quality: one phrase describing how light functions emotionally in this image
compositional_tension: one sentence on what creates visual energy or unease
photographic_tradition: the single closest named tradition or photographer aesthetic (e.g. Vivian Maier, Daido Moriyama, FSA documentary, New York School)
style_fingerprint: exactly 5 comma-separated descriptors for similarity search — choose from established photographic vocabulary, be specific enough to distinguish this image from others

FACTUAL DESCRIPTION:
{call1_output}"""


In [ ]:
# Cell: Parsers & Savers

BANNED = {
    "street", "urban", "city", "candid", "scene", "moment",
    "photography", "photo", "black and white", "monochrome",
    "bustling", "hustle",
}

_KV_KEY_MAP = {"contentlocation": "content_location"}


def parse_kv(raw: str) -> dict:
    result = {}
    for line in raw.splitlines():
        if ":" not in line:
            continue
        key, _, value = line.partition(":")
        key   = key.strip().lower()
        value = value.strip()
        if not key or not value:
            continue
        key = _KV_KEY_MAP.get(key, key)
        if key == "keywords":
            result[key] = [k.strip() for k in value.split(",") if k.strip()]
        else:
            result[key] = value
    return result


def clean_c1(parsed: dict) -> dict:
    parsed["keywords"] = list(dict.fromkeys([
        k for k in parsed.get("keywords", [])
        if k.lower() not in BANNED
    ]))
    if parsed.get("caption"):
        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', parsed["caption"]) if s.strip()]
        seen = set()
        unique = [s for s in sentences if s.lower() not in seen and not seen.add(s.lower())]
        parsed["caption"] = " ".join(unique)
    for field in ("name", "caption", "description"):
        val = parsed.get(field, "") or ""
        found = [w for w in BANNED if w in val.lower()]
        if found:
            print(f"  [warn] banned words in {field}: {found}", file=sys.stderr)
    return parsed


def save_call1(photo_id: str, run_id: int, raw: str, parsed: dict, latency_ms: int):
    con.execute("""
        INSERT INTO v3_outputs (photo_id, run_id)
        VALUES (?, ?)
        ON CONFLICT (photo_id, run_id) DO NOTHING
    """, [photo_id, run_id])
    con.execute("""
        UPDATE v3_outputs SET
            c1_raw           = ?,
            name             = ?,
            description      = ?,
            caption          = ?,
            keywords         = ?,
            content_location = ?,
            slug             = ?,
            confidence       = ?,
            c1_latency_ms    = ?
        WHERE photo_id = ? AND run_id = ?
    """, [
        raw,
        parsed.get("name"),
        parsed.get("description"),
        parsed.get("caption"),
        _json.dumps(parsed.get("keywords", [])),
        parsed.get("content_location"),
        parsed.get("slug"),
        int(parsed["confidence"]) if parsed.get("confidence") else None,
        latency_ms,
        photo_id, run_id,
    ])


def save_call2(photo_id: str, run_id: int, raw: str, parsed: dict, latency_ms: int):
    con.execute("""
        UPDATE v3_outputs SET
            c2_raw                 = ?,
            mood                   = ?,
            light_quality          = ?,
            compositional_tension  = ?,
            photographic_tradition = ?,
            style_fingerprint      = ?,
            c2_latency_ms          = ?
        WHERE photo_id = ? AND run_id = ?
    """, [
        raw,
        parsed.get("mood"),
        parsed.get("light_quality"),
        parsed.get("compositional_tension"),
        parsed.get("photographic_tradition"),
        parsed.get("style_fingerprint"),
        latency_ms,
        photo_id, run_id,
    ])

print("Parsers and savers ready.")


In [ ]:
# Cell: New Run
# Edit RUN_NOTES to describe what changed, then run this cell before the model cells.

RUN_NOTES = "v1 - initial 2-call Sonnet run"

next_id = con.execute("SELECT COALESCE(MAX(run_id), 0) + 1 FROM v3_runs").fetchone()[0]
con.execute(
    "INSERT INTO v3_runs (run_id, notes, c1_prompt, c2_prompt) VALUES (?, ?, ?, ?)",
    [next_id, RUN_NOTES, CALL1_PROMPT, CALL2_PROMPT],
)
CURRENT_RUN_ID = next_id
print(f"CURRENT_RUN_ID = {CURRENT_RUN_ID}  |  notes: {RUN_NOTES}")


In [ ]:
# Cell: Run Call 1 — Factual + SEO
# Produces: name, description, caption, keywords, content_location, slug, confidence

def run_call1(image_path: str) -> tuple[str, int]:
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    client = anthropic.Anthropic()
    t0 = time.time()
    response = client.messages.create(
        model=MODEL,
        max_tokens=700,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}},
                {"type": "text", "text": CALL1_PROMPT},
            ],
        }],
    )
    latency_ms = int((time.time() - t0) * 1000)
    return response.content[0].text.strip(), latency_ms


assert CURRENT_RUN_ID is not None, "Run the 'New Run' cell first to set CURRENT_RUN_ID"

done = {
    row[0] for row in
    con.execute(
        "SELECT photo_id FROM v3_outputs WHERE c1_raw IS NOT NULL AND run_id = ?",
        [CURRENT_RUN_ID],
    ).fetchall()
}

for photo_id, path in target_photos.items():
    if photo_id in done:
        print(f"  skip  {path.name}")
        continue
    raw, ms = run_call1(str(path))
    parsed = clean_c1(parse_kv(raw))
    save_call1(photo_id, CURRENT_RUN_ID, raw, parsed, ms)
    print(f"  done  {path.name} | {parsed.get('name','')[:60]} | conf={parsed.get('confidence')} | {ms}ms")

print("\nCall 1 complete.")


In [ ]:
# Cell: Run Call 2 — Artistic Analysis
# Produces: mood, light_quality, compositional_tension, photographic_tradition, style_fingerprint
# Requires Call 1 to be complete for each photo.

def run_call2(image_path: str, call1_output: str) -> tuple[str, int]:
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    client = anthropic.Anthropic()
    prompt = CALL2_PROMPT.format(call1_output=call1_output)
    t0 = time.time()
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}},
                {"type": "text", "text": prompt},
            ],
        }],
    )
    latency_ms = int((time.time() - t0) * 1000)
    return response.content[0].text.strip(), latency_ms


assert CURRENT_RUN_ID is not None, "Run the 'New Run' cell first to set CURRENT_RUN_ID"

c1_complete = {
    row[0] for row in
    con.execute(
        "SELECT photo_id FROM v3_outputs WHERE c1_raw IS NOT NULL AND run_id = ?",
        [CURRENT_RUN_ID],
    ).fetchall()
}
missing = [pid for pid in target_photos if pid not in c1_complete]
assert not missing, f"Call 1 incomplete for {len(missing)} photo(s). Run Call 1 first."

done = {
    row[0] for row in
    con.execute(
        "SELECT photo_id FROM v3_outputs WHERE c2_raw IS NOT NULL AND run_id = ?",
        [CURRENT_RUN_ID],
    ).fetchall()
}

for photo_id, path in target_photos.items():
    if photo_id in done:
        print(f"  skip  {path.name}")
        continue
    c1_raw = con.execute(
        "SELECT c1_raw FROM v3_outputs WHERE photo_id = ? AND run_id = ?",
        [photo_id, CURRENT_RUN_ID],
    ).fetchone()[0]
    raw, ms = run_call2(str(path), c1_raw)
    parsed = parse_kv(raw)
    save_call2(photo_id, CURRENT_RUN_ID, raw, parsed, ms)
    print(f"  done  {path.name} | {parsed.get('photographic_tradition','')[:50]} | mood={parsed.get('mood','')} | {ms}ms")

print("\nCall 2 complete.")


In [ ]:
# Cell: Visual Review

def show_review(photo_id: str, path):
    row = con.execute("""
        SELECT name, description, caption, keywords, content_location, confidence,
               mood, light_quality, compositional_tension,
               photographic_tradition, style_fingerprint,
               c1_latency_ms, c2_latency_ms
        FROM v3_outputs
        WHERE photo_id = ? AND run_id = ?
    """, [photo_id, CURRENT_RUN_ID]).fetchone()

    if row is None:
        print(f"No data for {path.name} in run {CURRENT_RUN_ID}")
        return

    (name, description, caption, keywords_json, location, confidence,
     mood, light_quality, comp_tension, tradition, style_fp,
     ms1, ms2) = row

    keywords = _json.loads(keywords_json) if keywords_json else []

    fig, axes = plt.subplots(1, 2, figsize=(15, 6),
                             gridspec_kw={"width_ratios": [1, 1.6]})
    img = Image.open(path)
    axes[0].imshow(img, cmap="gray" if img.mode == "L" else None)
    axes[0].axis("off")
    axes[0].set_title(path.name, fontsize=9, color="#666")

    lines = []
    lines.append(f"── Call 1 · Factual + SEO ({ms1}ms) ──────────────────")
    lines.append(f"name:       {name or '—'}")
    lines.append(f"desc:       {description or '—'}")
    lines.append(f"location:   {location or '—'}  conf={confidence or '—'}")
    if keywords:
        lines.append(f"keywords:   {', '.join(keywords[:6])}{'...' if len(keywords)>6 else ''}")
    lines.append("")
    lines.append(f"── Call 2 · Artistic ({ms2}ms) ────────────────────────")
    lines.append(f"mood:       {mood or '—'}")
    lines.append(f"light:      {light_quality or '—'}")
    lines.append(f"tension:    {comp_tension or '—'}")
    lines.append(f"tradition:  {tradition or '—'}")
    lines.append(f"style:      {style_fp or '—'}")

    axes[1].axis("off")
    axes[1].text(0.02, 0.98, "\n".join(lines),
                 transform=axes[1].transAxes,
                 va="top", ha="left", fontsize=9,
                 fontfamily="monospace")

    plt.tight_layout()
    plt.show()


assert CURRENT_RUN_ID is not None, "Run the 'New Run' cell first to set CURRENT_RUN_ID"

have = {
    row[0] for row in
    con.execute(
        "SELECT photo_id FROM v3_outputs WHERE c2_raw IS NOT NULL AND run_id = ?",
        [CURRENT_RUN_ID],
    ).fetchall()
}

reviewed = [pid for pid in hash_to_path if pid in have]
print(f"Showing {len(reviewed)} completed photos:")
for pid in reviewed:
    show_review(pid, hash_to_path[pid])


In [ ]:
# Cell: Analytics

assert CURRENT_RUN_ID is not None, "Run the 'New Run' cell first to set CURRENT_RUN_ID"

rows = con.execute("""
    SELECT photo_id,
           confidence, content_location,
           photographic_tradition, mood, style_fingerprint,
           c1_latency_ms, c2_latency_ms
    FROM v3_outputs
    WHERE run_id = ?
    ORDER BY photo_id
""", [CURRENT_RUN_ID]).fetchall()

if not rows:
    print("No data for this run yet. Run both calls first.")
else:
    total = len(rows)
    print(f"Run #{CURRENT_RUN_ID}  —  {total} photo(s) fully processed\n")

    # ── Field completeness ────────────────────────────────────────────────────
    fields = {
        "confidence": 1, "content_location": 2,
        "photographic_tradition": 3, "mood": 4, "style_fingerprint": 5,
    }
    print("Field completeness:")
    for field, idx in fields.items():
        filled = sum(1 for r in rows if r[idx])
        print(f"  {field:<26} {filled}/{total}  ({100*filled//total}%)")

    # ── Confidence distribution ───────────────────────────────────────────────
    from collections import Counter
    conf_counts = Counter(r[1] for r in rows if r[1])
    avg_conf = sum(int(k)*v for k,v in conf_counts.items()) / sum(conf_counts.values()) if conf_counts else 0
    print(f"\nConfidence avg: {avg_conf:.1f}  distribution: {dict(sorted(conf_counts.items()))}")

    # ── Photographic tradition breakdown ─────────────────────────────────────
    tradition_counts = Counter(r[3] for r in rows if r[3])
    print("\nPhotographic traditions:")
    for trad, cnt in tradition_counts.most_common():
        print(f"  {cnt:2d}x  {trad}")

    # ── Latency ───────────────────────────────────────────────────────────────
    ms1 = [r[6] for r in rows if r[6]]
    ms2 = [r[7] for r in rows if r[7]]
    if ms1:
        total_per_photo = (sum(ms1) + sum(ms2)) / total if ms2 else sum(ms1)/total
        print(f"\nLatency  Call1 avg={sum(ms1)//len(ms1)}ms  Call2 avg={sum(ms2)//len(ms2) if ms2 else '—'}ms  total/photo={int(total_per_photo)}ms")

    # ── Charts ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Confidence bar
    confs = [1, 2, 3, 4, 5]
    conf_vals = [conf_counts.get(str(c), 0) for c in confs]
    colors = ["#ef4444", "#f97316", "#eab308", "#22c55e", "#16a34a"]
    axes[0].bar([str(c) for c in confs], conf_vals, color=colors)
    axes[0].set_title(f"Confidence Distribution (avg {avg_conf:.1f})")
    axes[0].set_xlabel("Score")
    axes[0].set_ylabel("Photos")

    # Tradition breakdown
    if tradition_counts:
        trads = list(tradition_counts.keys())[:8]
        tcnts = [tradition_counts[t] for t in trads]
        axes[1].barh(trads, tcnts, color="#3b82f6")
        axes[1].set_title("Photographic Tradition")
        axes[1].set_xlabel("Photos")

    plt.tight_layout()
    plt.show()
